In [0]:
from pyspark.sql.functions import *
import re


df = spark.table("bronze.organizations")



#  fixing columns (snakecase + remove spaces/special chars)

df = df.toDF(*[
    re.sub(r'[^a-z0-9_]', '', c.lower().replace(" ", "_"))
    for c in df.columns
])

# 3. HANDLE NULL / UNKNOWN
# (replace blanks & 'unknown' with null)

df = df.replace(["", "unknown", "UNKNOWN"], None)

# zip should be numeric
df = df.withColumn("zip", col("zip").cast("long"))

# latitude & longitude → double
df = df.withColumn("lat", col("lat").cast("double")) \
       .withColumn("lon", col("lon").cast("double"))

# trim spaces
df = df.withColumn("name", trim(col("name"))) \
       .withColumn("address", trim(col("address"))) \
       .withColumn("city", trim(col("city")))

# standardize casing
df = df.withColumn("city", lower(col("city"))) \
       .withColumn("state", upper(col("state")))


# remove rows with missing id (critical field)
df = df.filter(col("id").isNotNull())

# read table
df = spark.table("silver.organizations")

# clean zip (remove spaces and convert to number)
df = df.withColumn("zip", trim(col("zip")).cast("long"))


# Remove system columns (like _line, _fivetran_synced)
df = df.select([
    col(c) for c in df.columns if not c.startswith("_")
])


# Save to silver
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.organizations")